# Delay Locked Loop Explainer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SiliconJackets/Cac_Spring26/blob/main/DelayLockedLoop.ipynb)

```
Copyright 2026 SiliconJackets @ Georgia Institute of Technology
SPDX-License-Identifier: Apache-2.0
```

|Name|Affiliation| Email |IEEE Member|SSCS Member|
|:--:|:----------:|:----------:|:----------:|:----------:|
|Ethan Huang|Georgia Institute of Technology|ethanhuang@gatech.edu|No|No|
|Member #2|Georgia Institute of Technology|[PLACEHOLDER]@gatech.edu|No|No|
|Member #3|Georgia Institute of Technology|[PLACEHOLDER]@gatech.edu|No|No|
|Member #4|Georgia Institute of Technology|[PLACEHOLDER]@gatech.edu|No|No|
|Member #5|Georgia Institute of Technology|[PLACEHOLDER]@gatech.edu|No|No|
|Member #6|Georgia Institute of Technology|[PLACEHOLDER]@gatech.edu|No|No|

This notebook provides a comprehensive explainer of our Delay Locked Loop (DLL) design, covering each major module: Phase Detector, Controller, DCDL, and Ring Oscillator. Furthermore, each major modules has multiple implementation variants, trade-off analysis, and simulation results. The design targets the [open source sky130 PDK](https://github.com/google/skywater-pdk/) by Google and Skywater.

## What is a Delay Locked Loop
---

A Delay Locked Loop (DLL) is a closed-loop feedback circuit that aligns the phase of an output clock to a reference clock by adjusting a digitally controlled delay line. Unlike a PLL, a DLL does not generate a new frequency. It shifts the phase of an existing clock until the output edge lines up with the reference edge. The core feedback loop consists of three blocks: a Phase Detector that compares the reference and feedback clocks and outputs directional error signals (`up`/`down`), a Controller that integrates those error signals into a digital control word, and a Digitally Controlled Delay Line (DCDL) that converts that control word into a physical propagation delay applied to the clock. A Ring Oscillator provides a tunable clock source for testing. The loop continuously corrects until the phase error is minimized, at which point the system is "locked."

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/dll_top_level.svg" width="700"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

This project explores multiple implementations of each block, allowing direct comparison of architectural trade-offs under identical test conditions. Each module section below describes the high-level function, presents each implementation variant with its pros/cons, and leaves space for simulation results and data analysis.

## Phase Detector
---

The phase detector is the sensory front-end of the DLL. It compares the rising edges of the reference clock (`clk_in`) and the feedback clock (`clk_out`) and produces two single-bit outputs: `up` (reference leads, loop should increase delay) and `down` (feedback leads, loop should decrease delay). It does not measure the magnitude of the phase error, only the direction. This bang-bang style of detection is simple and fully digital, but it means the loop can never settle perfectly still. Near lock, the detector continuously toggles, producing small limit-cycle oscillations that appear as jitter. The choice of phase detector architecture directly impacts the loop's accuracy, stability, and susceptibility to metastability.

![Phase Detector Waveforms](https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/waveform_viz/gifs/phase_detector.gif)

The waveform above shows the phase detector's `clk_in`, `clk_out`, `up`, and `down` signals over time. During acquisition the `up` or `down` signal is asserted persistently as the loop corrects the phase offset. As the loop approaches lock, the assertions become shorter and begin alternating, reflecting the limit-cycle behavior where the detector oscillates between "slightly early" and "slightly late" decisions each cycle.

---

### Implementation 1: Edge-Order Detector

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/pd_edge_order.svg" width="500"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

This is a "first-edge-wins" detector. Each clock edge tries to assert its corresponding output (`up` or `down`), but only succeeds if the other clock's edge has not already arrived. When `clk_in` rises first, `up` is set; when `clk_out` rises first, `down` is set. The opposite edge clears the output, completing the comparison for that cycle. The implementation uses two always blocks, each triggered on a different clock edge, with cross-checking logic to prevent both outputs from being asserted simultaneously.

Pros:
- Minimal logic, just two flip-flop-style blocks with cross-coupled gating
- No explicit reset feedback path needed

Cons:
- Race conditions when edges arrive nearly simultaneously; both `up` and `down` can briefly assert (observed in aligned-edge test cases)
- When `clk_out` leads `clk_in`, the detector can still incorrectly assert `up` because the first-edge-wins logic lets `clk_in` capture before `down` registers the earlier feedback edge
- Not robust enough for production use

> **TODO: Add simulation results / waveform figures for edge-order detector**

> **TODO: Add data analysis (measured accuracy across different phase offsets)**

---

### Implementation 2: Single Flip-Flop Detector

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/pd_single_ff.svg" width="460"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

The simplest possible phase detector. A single D flip-flop samples `clk_out` on the rising edge of `clk_in`. If `clk_out` is low at the sampling instant, the feedback edge hasn't arrived yet, so `clk_in` is leading and `up` is asserted. If `clk_out` is already high, feedback is ahead and `down` is asserted. There is no "equal" state; the detector always picks a direction.

Pros:
- Extremely low area and complexity, a single flip-flop plus output decode
- Fully synthesizable, fast timing

Cons:
- Inherently biased: when clocks are perfectly aligned, the detector still asserts `down` because `clk_out` is sampled as high at the rising edge of `clk_in`
- No frequency detection capability
- The bias means the loop locks with a small systematic phase offset rather than true zero error

> **TODO: Add simulation results / waveform figures for single-FF detector**

> **TODO: Add data analysis (bias characterization, jitter measurements)**

---

### Implementation 3: Phase-Frequency Detector (PFD)

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/pd_pfd.svg" width="500"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

The industry-standard architecture. Two flip-flops are held with D=1. The rising edge of `clk_in` sets the first FF (asserting `up`), and the rising edge of `clk_out` sets the second FF (asserting `down`). When both outputs are high simultaneously, an AND gate generates a reset pulse that clears both flip-flops. The width of the `up` or `down` pulse is proportional to the phase difference between the two clocks. This detector can also detect frequency mismatches: if one clock is faster, its corresponding output will be asserted for longer durations on average, driving the loop in the correct direction.

Pros:
- Detects both phase error and frequency error
- Pulse width encodes error magnitude (useful for analog loops, and provides clean direction for digital bang-bang use)
- Well-understood, widely used in PLL/DLL designs

Cons:
- Dead zone: when phase difference is very small, the reset pulse fires almost immediately, producing narrow output pulses that may not be captured reliably by downstream logic
- Slightly more complex than single-FF or edge-order approaches

> **TODO: Add simulation results / waveform figures for PFD**

> **TODO: Add data analysis (dead zone characterization, frequency acquisition behavior)**

---

### Implementation 4: Sampled XOR Detector

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/pd_xor.svg" width="540"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

An XOR gate produces a logic-1 whenever `clk_in` and `clk_out` differ, indicating a phase mismatch. This mismatch signal is then sampled by a D flip-flop on the rising edge of `clk_in`. If there is a mismatch and `clk_out` is low, the reference is leading (`up`). If `clk_out` is high, feedback is leading (`down`). When the clocks match at the sampling instant, both outputs are cleared.

Pros:
- Simple combinational front-end (single XOR gate)
- Naturally produces a "no error" state when clocks are aligned

Cons:
- XOR detects mismatch but not direction; direction is inferred by sampling `clk_out`, making it duty-cycle and timing dependent
- Biased toward detecting lag (`up`) while weak for detecting lead. When `clk_out` leads, the XOR term is often 0 at the `clk_in` sampling edge, so `down` is rarely asserted
- Not suitable as a standalone detector in most practical designs

> **TODO: Add simulation results / waveform figures for XOR detector**

> **TODO: Add data analysis (duty cycle sensitivity, directional bias measurements)**

## Controller
---

The controller sits between the phase detector and the DCDL. It receives the `up`/`down` signals and maintains an N-bit control word (`ctrl[N-1:0]`) that sets the delay through the DCDL. At its core, every controller variant is a digital accumulator: `ctrl[k+1] = ctrl[k] + up - down`. When `up` is asserted the control word increments (more delay), when `down` is asserted it decrements (less delay), and when both match it holds. The control word is clamped to `[0, 2^N - 1]` to prevent wrap-around. What differentiates the five implementations below is how aggressively they update. The step size, filtering, and mode-switching strategies each trade off lock speed against steady-state jitter. Because the phase detector only provides direction (not magnitude), the controller effectively sets the loop bandwidth: large steps mean fast convergence but more overshoot, small steps mean smooth tracking but slower acquisition.

![Controller Waveforms](https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/waveform_viz/gifs/controller.gif)

The waveform above shows `up`, `down`, and `ctrl[N-1:0]` evolving over time. During acquisition, `ctrl` ramps steadily as the controller integrates persistent `up` or `down` signals. Once the loop approaches lock, `ctrl` settles into a narrow range and begins toggling by small amounts, reflecting the limit-cycle behavior inherent to bang-bang control.

---

### Implementation 1: Saturating Controller

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/ctrl_saturating.svg" width="500"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

The baseline design. On each clock edge, the control word increments by 1 if `up` is asserted, decrements by 1 if `down` is asserted, and holds otherwise. Hard saturation clamps `ctrl` at 0 and `2^N - 1` to prevent underflow or overflow. On reset, `ctrl` initializes to a parameterized `INIT_CTRL` value (default midpoint of 32 for 6-bit control). The entire update path is a single add/subtract with a compare, one combinational level plus the output register.

Pros:
- Simplest possible controller, easy to verify and reason about
- Predictable, monotonic response to persistent error
- Industry baseline for comparison

Cons:
- Fixed ±1 step means slow convergence when far from lock
- Near lock, the ±1 toggling produces a limit cycle with 1-LSB jitter that cannot be reduced without external filtering

> **TODO: Add simulation results / waveform figures for saturating controller**

> **TODO: Add data analysis (lock time, steady-state jitter amplitude)**

---

### Implementation 2: Filtered Controller

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/ctrl_filtered.svg" width="580"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

Instead of updating `ctrl` on every cycle, this controller requires `FILTER_LEN` consecutive `up` (or `down`) requests before making a single ±1 adjustment. Two internal counters track consecutive `up` and `down` assertions independently. If the direction changes or both signals are idle, the counters reset to zero. Only when one counter reaches the threshold does `ctrl` update, and then that counter resets. This acts as a simple digital low-pass filter on the phase detector output.

Pros:
- Significantly reduces jitter near lock; transient noise or single-cycle glitches from the phase detector are ignored
- Steady-state limit cycle is suppressed because the toggling `up`/`down` pattern resets the counters before threshold is reached

Cons:
- Slower lock acquisition, the loop must see `FILTER_LEN` consistent cycles before taking any action
- The `FILTER_LEN` parameter requires tuning: too small and it behaves like the saturating controller, too large and the loop becomes unresponsive

> **TODO: Add simulation results / waveform figures for filtered controller**

> **TODO: Add data analysis (lock time vs FILTER_LEN, jitter reduction compared to saturating)**

---

### Implementation 3: Acquire/Track Controller

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/ctrl_acquire_track.svg" width="580"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

A dual-mode controller. It starts in acquire mode with a large step size (`ACQUIRE_STEP`, default 4) for fast convergence. A quiet counter tracks consecutive idle cycles (no `up` or `down` assertion). Once `QUIET_CYCLES` (default 8) consecutive quiet cycles are observed, indicating the loop is near lock, the controller switches to track mode with a small step size (`TRACK_STEP`, default 1) to minimize jitter. Any new `up` or `down` assertion resets the quiet counter but does not revert the mode. The mode transition is one-way: once in track mode, the controller stays there until reset.

Pros:
- Fast acquisition (large steps cover the control range quickly) with low steady-state jitter (small steps near lock)
- Widely used in industry DLL/PLL designs as a practical balance of speed and stability

Cons:
- One-way mode switch means the controller cannot re-acquire if the loop is disturbed after locking (e.g., supply noise kicks the delay far from lock)
- The `QUIET_CYCLES` threshold must be tuned; too aggressive and the controller switches to track mode before actually reaching lock

> **TODO: Add simulation results / waveform figures for acquire/track controller**

> **TODO: Add data analysis (acquisition phase duration, jitter comparison between modes)**

---

### Implementation 4: Coarse/Fine Controller

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/ctrl_coarse_fine.svg" width="580"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

This controller splits the control word into two fields: an upper coarse field (`COARSE_BITS`, default 3) and a lower fine field (`FINE_BITS`, default 3), concatenated as `ctrl = {coarse, fine}`. It starts in coarse mode, adjusting only the coarse bits for large delay jumps. After `SWITCH_QUIET` (default 8) consecutive quiet cycles, it transitions to fine mode, where only the fine bits are adjusted for precise tuning. Each field saturates independently at its own bounds. This effectively gives the controller two levels of resolution in a single control word.

Pros:
- High total resolution without needing an extremely wide control word; coarse bits provide range, fine bits provide precision
- Clean separation of acquisition (coarse) and tracking (fine) phases
- Hardware-efficient: the two counters are small and independent

Cons:
- More complex than a single-counter design, with two independent saturating counters plus mode logic
- Coarse-to-fine boundary can introduce a jump if the coarse step doesn't land near the correct fine range
- Same one-way mode switch limitation as acquire/track

> **TODO: Add simulation results / waveform figures for coarse/fine controller**

> **TODO: Add data analysis (resolution improvement, coarse-to-fine transition behavior)**

---

### Implementation 5: Variable-Step Controller

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/ctrl_variable_step.svg" width="580"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

An adaptive controller where the step size grows with repeated error in the same direction. A 4-bit counter (`same_dir_count`) tracks how many consecutive cycles the phase detector has requested the same direction. When the count exceeds `MED_THRESH` (default 4), the step size increases from 1 to `MED_STEP` (default 2). When it exceeds `BIG_THRESH` (default 8), it jumps to `BIG_STEP` (default 4). A direction change resets the counter to 1. This creates nonlinear, accelerating behavior: the loop crawls when error is small (suggesting near-lock) and sprints when error is persistent (suggesting far from lock).

Pros:
- Fastest convergence of all five controllers, accelerates when far from lock without requiring an explicit mode switch
- Naturally adapts to both acquisition and tracking without separate modes

Cons:
- More tuning parameters (`MED_THRESH`, `BIG_THRESH`, `MED_STEP`, `BIG_STEP`); poor tuning can cause overshoot or oscillation
- Near lock, even a short run of consistent `up` or `down` from the limit cycle can trigger medium steps, potentially increasing jitter compared to a fixed ±1 controller

> **TODO: Add simulation results / waveform figures for variable-step controller**

> **TODO: Add data analysis (convergence speed comparison, overshoot characterization)**

## DCDL (Digitally Controlled Delay Line)
---

The DCDL is the actuator of the DLL. It takes the digital control word from the controller and converts it into a physical propagation delay applied to the clock signal. The input signal `A` enters a chain of delay elements, and the control word `Q` selects how many stages the signal passes through (or which delayed tap is routed to the output `Y`). A larger control value means more delay stages are active, producing a longer delay. The DCDL must provide monotonic, glitch-free delay adjustment across its full control range. Different architectures achieve this using different delay primitives (inverters vs. NAND gates), selection strategies (mux trees vs. bypass logic), and polarity-correction techniques. The five implementations below span the design space from simple inverter chains to dual-chain vernier interpolation.

---

### Implementation 1: Inverter-Based DCDL

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/dcdl_inverter.svg" width="660"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

> **TODO: Generate waveform GIF for inverter-based DCDL with correct timings**

A 4-stage delay line where each stage consists of two inverters in series (preserving signal polarity). The input `A` propagates through this chain, producing four delayed taps (`tap0` through `tap3`). A 2-bit control word `Q` selects among these taps using a hierarchical mux tree: two first-level muxes each choose between a pair of adjacent taps based on `Q[0]`, and a final mux selects between the two first-level outputs based on `Q[1]`. This gives four selectable delay values with clean, non-inverted output at every tap.

Pros:
- Straightforward design; the double-inverter stages guarantee consistent polarity at every tap
- Hierarchical mux tree scales cleanly with more stages
- Easy to reason about delay linearity

Cons:
- Two inverters per stage means more area and power per delay step
- Mux tree adds its own propagation delay to every path, which may limit minimum achievable delay
- 2-bit control gives only 4 delay steps, providing coarse resolution

> **TODO: Add simulation results / waveform figures for inverter-based DCDL**

> **TODO: Add data analysis (delay per step, linearity measurement)**

---

### Implementation 2: Conditional Inverter DCDL

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/dcdl_conditional.svg" width="600"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

> **TODO: Generate waveform GIF for conditional inverter DCDL with correct timings**

This design uses a single inverter per stage instead of two, halving the chain length for the same number of taps. The trade-off is that odd-numbered taps produce an inverted signal. A mux tree identical to Implementation 1 selects the desired tap, and then an XNOR gate at the output corrects the polarity: `Y = mux_out XNOR Q[1]`. When `Q[1]` is 0 the output passes through unchanged; when `Q[1]` is 1 the XNOR inverts it, compensating for the odd number of inversions in that path.

Pros:
- Half the inverters of Implementation 1 for the same number of taps, meaning smaller area and lower power
- Finer delay granularity per stage (each inverter adds one gate delay instead of two)

Cons:
- The XNOR correction gate adds a fixed delay to every output path, and its own delay variation can affect linearity
- The polarity trick only works cleanly when the mux select bits align with the inversion pattern; extending to wider control words requires careful bookkeeping

> **TODO: Add simulation results / additional waveform figures for conditional inverter DCDL**

> **TODO: Add data analysis (delay per step vs. Implementation 1, area comparison)**

---

### Implementation 3: Glitch-Free Inverter DCDL

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/dcdl_glitch_free.svg" width="620"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

> **TODO: Generate waveform GIF for glitch-free DCDL with correct timings**

This variant addresses glitch hazards that can occur when the mux select changes while the signal is propagating. Instead of a mux tree, each delayed tap is inverted and then gated with a one-hot select signal using a NAND2 gate. The one-hot select is registered on the clock edge (`sel_reg`), so it only changes synchronously, eliminating combinational glitches during select transitions. The gated tap outputs are combined through a NAND tree (`y0*y1` and `y2*y3`, then the two results combined) to produce the final output. On reset, `sel_reg` defaults to selecting `tap0`.

Pros:
- Glitch-free by construction; the registered one-hot select ensures no transient paths are created during control word changes
- NAND-based gating and combining is area-efficient in standard cell libraries

Cons:
- Requires a clock input (`clk`) and reset (`rst_n`), unlike the purely combinational implementations above
- The registered select adds one cycle of latency between a control word change and the delay actually updating
- The NAND tree path adds fixed delay overhead

> **TODO: Add simulation results / additional waveform figures for glitch-free DCDL**

> **TODO: Add data analysis (glitch characterization vs. Implementations 1 and 2)**

---

### Implementation 4: NAND-Based DCDL

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/dcdl_nand.svg" width="640"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

> **TODO: Generate waveform GIF for NAND-based DCDL with correct timings**

A fundamentally different approach: instead of selecting a tap from a shared chain, each stage is a NAND-based delay cell that can either propagate or bypass the signal. Each cell has three NAND2 gates and one inverter. `nand2_1` gates the propagated input (`in1`) with the inverted control bit (`~ctl`), `nand2_2` gates the bypass input (`in0`) with `ctl`, and `nand2_3` combines the two. When `ctl=1`, the cell passes `in0` (bypass); when `ctl=0`, it passes `in1` (propagated from the previous stage). The 4-bit control word `Q[3:0]` independently enables each of the four stages, allowing the delay to scale from 0 active stages to 4.

Pros:
- Wider delay range per stage than inverter-based designs, since each NAND cell introduces more gate delay than a single inverter
- Independent per-stage control allows non-sequential activation patterns
- `dont_touch` and `keep` annotations prevent the synthesizer from optimizing away the intentional delay structure

Cons:
- Non-linear delay steps; the delay added by enabling one more NAND cell depends on the cell's position in the chain and the state of other cells
- More complex per-cell logic (3 NAND + 1 inverter per stage vs. 1-2 inverters)
- Thermometer or one-hot encoding of `Q` may be needed for monotonic delay if the stages are not identical

> **TODO: Add simulation results / additional waveform figures for NAND-based DCDL**

> **TODO: Add data analysis (delay range, per-step nonlinearity)**

---

### Implementation 5: Vernier Crossover DCDL

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/dcdl_vernier.svg" width="660"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

The most sophisticated architecture, targeting fine delay resolution through interpolation between two delay chains with different per-stage delays. A "slow" chain uses `sky130_fd_sc_hd__buf_1` cells (small, slow buffers) and a "fast" chain uses `sky130_fd_sc_hd__buf_4` cells (large, fast buffers). At each of the 16 stages, a 2:1 mux controlled by `Q[i]` can cross the signal from the slow chain into the fast chain. The input `A` enters the slow chain; the fast chain starts at logic-0. When `Q[i]` is asserted at stage `i`, the slow-chain signal crosses over to the fast chain at that point and propagates through the remaining fast-chain stages. The output is taken from the end of the fast chain. The total delay is determined by how many slow stages the signal traverses before crossing over to the fast chain, so more slow stages means more delay.

Pros:
- Very fine delay resolution; the delay step is the difference between one `buf_1` delay and one `buf_4` delay, which can be much smaller than either buffer's absolute delay
- 16-stage design with per-stage control gives high granularity
- Uses sky130 standard cells directly, ready for physical implementation

Cons:
- Largest area and power of all five implementations, with two full buffer chains plus 16 muxes
- Delay linearity depends on matching between the `buf_1` and `buf_4` cell delays across PVT corners
- The `fast_net[0] = 1'b0` initialization means the fast chain output is invalid until at least one crossover mux is enabled

> **TODO: Add simulation results / waveform figures for vernier DCDL**

> **TODO: Add data analysis (delay resolution measurement, linearity across PVT corners)**

## Ring Oscillator
---

The ring oscillator is not part of the DLL feedback loop itself. It serves as a tunable on-chip clock source for testing and characterization. It generates a free-running clock by feeding the output of an odd-length inverter chain back to its input, creating a self-sustaining oscillation. The oscillation frequency is determined by the total propagation delay around the loop: `f = 1 / (2 * N * t_inv)`, where `N` is the number of inverter stages and `t_inv` is the delay per inverter. This implementation provides frequency selection by tapping the chain at different points.

---

### Implementation 1: Selectable-Length Inverter Ring

<img src="https://raw.githubusercontent.com/SiliconJackets/Cac_Spring26/main/diagrams/ring_oscillator.svg" width="620"/>

> **TODO: Replace above SVG with a proper draw.io diagram**

A chain of 9 inverters (inv0 through inv8) is driven from the output `clk_out` back through the chain. A 2-bit select input `sel[1:0]` chooses which tap feeds back to `clk_out` via a combinational case statement:

- `sel = 00` : feedback from `n2` (3 inverters in the loop, highest frequency)
- `sel = 01` : feedback from `n4` (5 inverters, lower frequency)
- `sel = 10` : feedback from `n6` (7 inverters, lower still)
- `sel = 11` : feedback from `n8` (9 inverters, lowest frequency)

Each configuration uses an odd number of inversions, ensuring sustained oscillation. Shorter loops oscillate faster because the total round-trip delay is smaller.

Pros:
- Simple, compact design, just inverters and a mux
- Provides four discrete frequency options from a single oscillator
- No external clock input required, useful for standalone testing of the DCDL or full DLL loop

Cons:
- Only one implementation, so no architectural comparison for this module
- Frequency is highly sensitive to PVT (process, voltage, temperature) variation since it depends entirely on inverter delay
- The `always_comb` feedback style may cause simulation issues in some tools (zero-delay oscillation in RTL sim without gate-level timing)
- Frequency selection is coarse, with only four options and large gaps between them

> **TODO: Add simulation results / waveform figures for ring oscillator at each sel value**

> **TODO: Add data analysis (measured frequency vs. sel setting, PVT sensitivity if available)**

## Practical Examples
---